<a href="https://colab.research.google.com/github/Hadeel-93/HW-8/blob/main/HW_8_Q2_weightdecay.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader


In [3]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using device:", device)
    print("GPU:", torch.cuda.get_device_name(0))
    # GPU performance
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.cuda.empty_cache()
else:
    device = torch.device("cpu")
    print("Using device: CPU (GPU not available)")

# Reproducibility
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

Using device: CPU (GPU not available)


In [4]:
data_path = "/content/data-cifar10/"

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465],
        std=[0.2470, 0.2435, 0.2616]
    )
])

train_dataset = datasets.CIFAR10(
    root=data_path,
    train=True,
    download=True,
    transform=transform
)

eval_dataset = datasets.CIFAR10(
    root=data_path,
    train=False,
    download=True,
    transform=transform
)

print("Training images:", len(train_dataset))
print("Evaluation images:", len(eval_dataset))
print("Classes:", train_dataset.classes)

100%|██████████| 170M/170M [13:07<00:00, 217kB/s]


Training images: 50000
Evaluation images: 10000
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [6]:
BATCH_SIZE = 128
NUM_WORKERS = 2

generator = torch.Generator()
generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    generator=generator
)

eval_loader = DataLoader(
    eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

print("Training batches:", len(train_loader))
print("Evaluation batches:", len(eval_loader))

Training batches: 391
Evaluation batches: 79


In [7]:
class ResidualBlock(nn.Module):

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=True
        )

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=True
        )

        # Skip connection
        if stride != 1 or in_channels != out_channels:

            self.shortcut = nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                stride=stride,
                bias=True
            )

        else:
            self.shortcut = nn.Identity()

    def forward(self, x):

        identity = self.shortcut(x)

        out = F.relu(self.conv1(x))
        out = self.conv2(out)

        out = out + identity

        out = F.relu(out)

        return out


In [8]:
class ResNet10(nn.Module):

    def __init__(self):
        super().__init__()

        # Initial convolution
        self.conv1 = nn.Conv2d(
            3,
            64,
            kernel_size=3,
            stride=1,
            padding=1
        )

        self.in_channels = 64

        # Stage 1: 2 residual blocks
        self.layer1 = self.make_layer(
            64, num_blocks=2, stride=1
        )

        # Stage 2: 2 residual blocks
        self.layer2 = self.make_layer(
            128, num_blocks=2, stride=2
        )

        # Stage 3: 2 residual blocks
        self.layer3 = self.make_layer(
            256, num_blocks=2, stride=2
        )

        # Stage 4: 2 residual blocks
        self.layer4 = self.make_layer(
            512, num_blocks=2, stride=2
        )

        # Stage 5: 2 residual blocks
        self.layer5 = self.make_layer(
            512, num_blocks=2, stride=1
        )

        # Global average pooling
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))

        # Classifier
        self.fc = nn.Linear(512, 10)

    def make_layer(self, out_channels, num_blocks, stride):

        layers = []

        layers.append(
            ResidualBlock(
                self.in_channels,
                out_channels,
                stride
            )
        )

        self.in_channels = out_channels

        for _ in range(1, num_blocks):

            layers.append(
                ResidualBlock(
                    self.in_channels,
                    out_channels
                )
            )

        return nn.Sequential(*layers)

    def forward(self, x):

        x = F.relu(self.conv1(x))

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)

        x = self.avg_pool(x)

        x = torch.flatten(x, 1)

        x = self.fc(x)

        return x


In [9]:
model = ResNet10().to(device)

def count_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

# Verify number of residual blocks
num_blocks = sum(
    1 for module in model.modules()
    if isinstance(module, ResidualBlock)
)

print("Number of residual blocks:", num_blocks)
print("Trainable parameters:", count_parameters(model))

# Test model output
sample_images, sample_labels = next(iter(train_loader))
sample_images = sample_images.to(device)

with torch.no_grad():
    sample_outputs = model(sample_images)

print("Input shape:", sample_images.shape)
print("Output shape:", sample_outputs.shape)

assert num_blocks == 10
assert sample_outputs.shape[1] == 10

print("ResNet-10 architecture verified successfully.")


Number of residual blocks: 10
Trainable parameters: 20608394


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Input shape: torch.Size([128, 3, 32, 32])
Output shape: torch.Size([128, 10])
ResNet-10 architecture verified successfully.


In [10]:
NUM_EPOCHS = 300
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.001

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scaler = torch.amp.GradScaler("cuda")

print("Optimizer: Adam")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Epochs:", NUM_EPOCHS)


Optimizer: Adam
Learning rate: 0.001
Weight decay: 0.001
Epochs: 300


/tmp/ipykernel_3389/3404988013.py:13: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  scaler = torch.amp.GradScaler("cuda")


In [11]:
def train_one_epoch():

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)

        predicted = outputs.argmax(dim=1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / total
    train_accuracy = correct / total

    return train_loss, train_accuracy


In [12]:
def evaluate_model():

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in eval_loader:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                outputs = model(images)
                loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            predicted = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    eval_loss = running_loss / total
    eval_accuracy = correct / total

    return eval_loss, eval_accuracy


In [ ]:
train_losses = []
train_accuracies = []

eval_losses = []
eval_accuracies = []

epoch_times = []

if torch.cuda.is_available():
    torch.cuda.synchronize()
total_start_time = time.time()

for epoch in range(NUM_EPOCHS):

    epoch_start_time = time.time()

    # Training
    train_loss, train_accuracy = train_one_epoch()

    # Evaluation
    eval_loss, eval_accuracy = evaluate_model()

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    epoch_time = time.time() - epoch_start_time

    # Save results
    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)

    eval_losses.append(eval_loss)
    eval_accuracies.append(eval_accuracy)

    epoch_times.append(epoch_time)

    print(
        f"Epoch [{epoch + 1}/{NUM_EPOCHS}] "
        f"| Time: {epoch_time:.2f}s "
        f"| Train Loss: {train_loss:.4f} "
        f"| Train Acc: {train_accuracy * 100:.2f}% "
        f"| Eval Loss: {eval_loss:.4f} "
        f"| Eval Acc: {eval_accuracy * 100:.2f}%"
    )

if torch.cuda.is_available():
    torch.cuda.synchronize()
total_training_time = time.time() - total_start_time

print("\nTraining completed!")

/tmp/ipykernel_3389/1462514527.py:16: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with torch.autocast(


In [ ]:
final_train_loss = train_losses[-1]
final_train_accuracy = train_accuracies[-1] * 100

final_eval_loss = eval_losses[-1]
final_eval_accuracy = eval_accuracies[-1] * 100

total_training_minutes = total_training_time / 60

num_parameters = count_parameters(model)

print("========================================")
print("RESNET-10 WITH WEIGHT DECAY RESULTS")
print("========================================")

print("Epochs:", NUM_EPOCHS)

print(
    f"Total Training Time: "
    f"{total_training_minutes:.2f} minutes"
)

print(
    f"Final Training Loss: "
    f"{final_train_loss:.4f}"
)

print(
    f"Final Training Accuracy: "
    f"{final_train_accuracy:.2f}%"
)

print(
    f"Final Evaluation Loss: "
    f"{final_eval_loss:.4f}"
)

print(
    f"Final Evaluation Accuracy: "
    f"{final_eval_accuracy:.2f}%"
)

print("Trainable Parameters:", num_parameters)

print(
    f"Average Epoch Time: "
    f"{np.mean(epoch_times):.2f} seconds"
)


In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(train_losses, label="Training Loss")
plt.plot(eval_losses, label="Evaluation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("ResNet-10 with Weight Decay: Loss")

plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    np.array(train_accuracies) * 100,
    label="Training Accuracy"
)

plt.plot(
    np.array(eval_accuracies) * 100,
    label="Evaluation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("ResNet-10 with Weight Decay: Accuracy")

plt.legend()
plt.grid(True)
plt.show()
